# Create Lite Dataset for Real-Time Demo

Sample a small, balanced subset from the full cleaned dataset for fast inference demos.

**Why a lite dataset?**
- The full dataset (~39k rows) is too large for quick testing and live demonstrations
- A balanced 2,000-row subset loads in milliseconds and keeps demos responsive
- Useful for rapid iteration during development, unit tests, and classroom presentations

**How it will be used:**
- Fed into the saved DistilBERT model for real-time prediction demos
- Powers a Streamlit / Gradio front-end where users paste a headline and see the result instantly
- Serves as a quick validation set to sanity-check model behaviour after loading

In [1]:
import pandas as pd

## 1. Load Full Cleaned Dataset

In [2]:
FULL_PATH = "data/fake_news_full_clean.csv"

df = pd.read_csv(FULL_PATH)

print(f"Full dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df["label"].value_counts().rename({0: "Real (0)", 1: "Fake (1)"}))

Full dataset shape: (39103, 6)
Columns: ['title', 'text', 'content', 'subject', 'date', 'label']

Label distribution:
label
Real (0)    21196
Fake (1)    17907
Name: count, dtype: int64


## 2. Balanced Sampling (1,000 per class)

In [3]:
SAMPLES_PER_CLASS = 1000
RANDOM_STATE = 42

samples = []

for label_val, label_name in [(1, "Fake"), (0, "Real")]:
    subset = df[df["label"] == label_val]
    available = len(subset)

    n = min(SAMPLES_PER_CLASS, available)
    if available < SAMPLES_PER_CLASS:
        print(f"  ⚠ {label_name} has only {available} samples (< {SAMPLES_PER_CLASS}), using all.")

    sampled = subset.sample(n=n, random_state=RANDOM_STATE)
    samples.append(sampled)
    print(f"  {label_name}: sampled {n} / {available}")

df_lite = pd.concat(samples, ignore_index=True)
print(f"\nCombined lite size before shuffle: {len(df_lite)}")

  Fake: sampled 1000 / 17907
  Real: sampled 1000 / 21196

Combined lite size before shuffle: 2000


## 3. Shuffle

In [4]:
df_lite = df_lite.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Shuffled lite dataset shape: {df_lite.shape}")

Shuffled lite dataset shape: (2000, 6)


## 4. Validation

In [5]:
print(f"Shape: {df_lite.shape}")
print(f"Columns: {df_lite.columns.tolist()}")

print(f"\nClass distribution:")
print(df_lite["label"].value_counts().rename({0: "Real (0)", 1: "Fake (1)"}))

print(f"\n--- Sample rows ---")
df_lite[["content", "label"]].head(5)

Shape: (2000, 6)
Columns: ['title', 'text', 'content', 'subject', 'date', 'label']

Class distribution:
label
Real (0)    1000
Fake (1)    1000
Name: count, dtype: int64

--- Sample rows ---


,content,label
0,"In Georgia, costliest U.S. House race hits ugl...",0
1,"Bernie Gives Us Hope, Promises To Protect Mino...",1
2,U.S. appeals court questions scope of Trump tr...,0
3,Trump ‘Promises’ To Fix Black People’s Problem...,1
4,Countdown to Brexit breakthrough? BRUSSELS (Re...,0


In [6]:
assert len(df_lite) == 2 * SAMPLES_PER_CLASS, "Unexpected row count!"
assert set(df_lite["label"].unique()) == {0, 1}, "Missing class!"
assert df_lite["content"].isnull().sum() == 0, "Null content found!"
assert set(df_lite.columns) == set(df.columns), "Column mismatch with full dataset!"
print("All validation checks passed.")

All validation checks passed.


## 5. Save Lite Dataset

In [7]:
LITE_PATH = "data/fake_news_lite_clean.csv"

df_lite.to_csv(LITE_PATH, index=False)

df_verify = pd.read_csv(LITE_PATH)
print(f"Saved to: {LITE_PATH}")
print(f"Verified shape: {df_verify.shape}")
print(f"Verified labels: {dict(df_verify['label'].value_counts())}")

Saved to: data/fake_news_lite_clean.csv
Verified shape: (2000, 6)
Verified labels: {0: np.int64(1000), 1: np.int64(1000)}


## Summary

| Item | Detail |
|------|--------|
| Source | `data/fake_news_full_clean.csv` (39,103 rows) |
| Sampling | 1,000 Fake + 1,000 Real (balanced) |
| Random seed | 42 |
| Edge cases | Caps at available count if < 1,000 |
| Output | `data/fake_news_lite_clean.csv` (2,000 rows) |
| Structure | Identical columns to full dataset |